In [23]:
import numpy as np
data=[]
with open('dataset.txt','r') as f:
    data=f.readlines()

for i in range(len(data)):
    data[i]=data[i].split()
data=np.array(data)
y=np.float64(data[:,-1])
X=np.float64(data[:,0:data.shape[1]-1])
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.8,random_state=42)

In [24]:

X = np.array([
    [0.2, 0.3, 0.1],
    [0.4, 0.1, 0.2],
    [0.6, 0.4, 0.3],
    [0.8, 0.7, 0.4],
    [1.0, 0.9, 0.5]
], dtype=float)

y = np.array([0.25, 0.35, 0.55, 0.75, 0.95], dtype=float)
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.8,random_state=42)


In [25]:
X.shape[1]

3

In [ ]:
import numpy as np
dim=[X.shape[1],3,2,1]

layers=len(dim)-1

weights={}
bais={}

for i in range(1,layers+1):
    weights['w'+str(i)]=np.random.rand(dim[i-1],dim[i])
    bais['b'+str(i)]=np.random.rand(dim[i],1)

In [ ]:
def dif_activation(layer,outputs):
    A=[]
    for i in outputs[f"o{layer}"][0]:
        if i>0:
            A.append(1)
        else:
            A.append(0)
    A=np.array([A])
    return A

In [28]:
def activation(inp):
    #relu activation
    A=[]
    for i in inp[0]:
        if i>=0:
            A.append(i)
        else:
            A.append(0)
    return np.array([A])

In [29]:
#forward_propagation
def forward_propagation(x,weights,bais):
    outputs={}
    outputs['o0']=[x]
    for i in range(1,layers+1):
        outputs['o'+str(i)]=activation((np.dot(weights['w'+str(i)].T,np.array(outputs['o'+str(i-1)]).T)+bais['b'+str(i)]).reshape(1,-1))
    return outputs['o'+str(layers)][0][0],outputs

In [30]:
def deepcopy(d):
    temp={}
    for i in d:
        temp[i]=d[i].copy()
    return temp

In [33]:
def back_propagation(weights,bais,y_hat,y,outputs,learning_rate):
    weights_=deepcopy(weights)
    bais_=deepcopy(bais)
    n=len(outputs)-1
    back=np.array([[-2*(y-y_hat)]])
    for j in range(n,0,-1):
        weights['w'+str(j)]=weights['w'+str(j)]-learning_rate*(np.dot(np.array(outputs['o'+str(j-1)]).T,back))
        bais[f"b{j}"]=bais[f"b{j}"]-learning_rate*(back.T)
        if j!=1:
            dif_a=dif_activation(j-1,outputs)
            back=np.diag(np.dot(dif_a.T,np.dot(back,weights_[f"w{j}"].T)))
            back=back.reshape(1,-1)
    del weights_
    del bais_
    return

In [34]:
import tqdm
epochs=5
learning_rate=0.01
for i in range(epochs):
    loss=[]
    print(f"epoch{i+1} :",end=" ")
    for j in tqdm.tqdm(range(X_train.shape[0])):
        ind=np.random.randint(0,X_train.shape[0])
        x=np.array([np.float64(i) for i in X_train[ind]])
        pred=forward_propagation(x,weights,bais)
        y_hat=pred[0]
        outputs=pred[1]
        loss.append((y[ind]-y_hat)**2)
        back_propagation(weights,bais,y_hat,y[ind],outputs,learning_rate)
    print(np.array(loss).mean())

epoch1 : 

100%|██████████| 4/4 [00:00<00:00, 2496.24it/s]


1.6052891529255153
epoch2 : 

100%|██████████| 4/4 [00:00<00:00, 2442.10it/s]


0.8830330598223343
epoch3 : 

...

0.007269601510257068
epoch4 : 

100%|██████████| 4/4 [00:00<00:00, 3278.08it/s]


0.19715754170892508
epoch5 : 

100%|██████████| 4/4 [00:00<00:00, 3129.49it/s]

0.18885843072378722


In [ ]:
import numpy as np
import tqdm
class Neural_network:
    def __init__(self,dim,learning_rate):
        self.learning_rate=learning_rate
        dim=dim
        self.dim=dim
        layers=len(dim)-1
        self.layers=layers
        weights={}
        bais={}
        for i in range(1,layers+1):
            weights['w'+str(i)]=np.random.rand(dim[i-1],dim[i])
            bais['b'+str(i)]=np.random.rand(dim[i],1)
        self.weights=weights
        self.bais=bais
    
    def forward_propagation(self,x):
        outputs={}
        outputs['o0']=[x]
        weights=self.weights
        bais=self.bais
        for i in range(1,self.layers+1):
            outputs['o'+str(i)]=(np.dot(weights['w'+str(i)].T,np.array(outputs['o'+str(i-1)]).T)+bais['b'+str(i)]).reshape(1,-1)
        return outputs['o'+str(self.layers)][0][0],outputs
    
    def deepcopy(self,d):
        temp={}
        for i in d:
            temp[i]=d[i].copy()
        return temp
    
    def back_propagation(self,y_hat,y,outputs):
        weights=self.weights
        bais=self.bais
        learning_rate=self.learning_rate
        weights_=self.deepcopy(weights)
        bais_=self.deepcopy(bais)
        n=len(outputs)-1
        back=np.array([[-2*(y-y_hat)]])
        for j in range(n,0,-1):
            weights['w'+str(j)]=weights['w'+str(j)]-learning_rate*(np.dot(np.array(outputs['o'+str(j-1)]).T,back))
            bais[f"b{j}"]=bais[f"b{j}"]-learning_rate*(back.T)
            if j!=1:
                back=np.dot(back,weights_[f"w{j}"].T)
        del weights_
        del bais_
        return
    
    def fit(self,X_train,y_train,epochs=5):

        for i in range(epochs):
            loss=[]
            print(f"epoch{i+1} :",end=" ")
            for j in tqdm.tqdm(range(X_train.shape[0])):
                ind=np.random.randint(0,X_train.shape[0])
                x=np.array([np.float64(i) for i in X_train[ind]])
                pred=self.forward_propagation(x)
                y_hat=pred[0]
                outputs=pred[1]
                loss.append((y_train[ind]-y_hat)**2)
                self.back_propagation(y_hat,y_train[ind],outputs)
            print("  loss:",np.array(loss).mean())
    def predict(self,X_test):
        y_pred=[]
        for i in X_test:
            y_pred.append(self.forward_propagation(i)[0])
        return np.array(y_pred)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.array([
    [0.2, 0.3, 0.1],
    [0.4, 0.1, 0.2],
    [0.6, 0.4, 0.3],
    [0.8, 0.7, 0.4],
    [1.0, 0.9, 0.5]
], dtype=float)

y = np.array([0.25, 0.35, 0.55, 0.75, 0.95], dtype=float)
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.8,random_state=42)

ann=Neural_network([X.shape[1],3,2,1],0.01)
ann.fit(X_train,y_train,5)

epoch1 : 

100%|██████████| 4/4 [00:00<00:00, 4892.74it/s]


  loss: 2.6368551838902765
epoch2 : 

100%|██████████| 4/4 [00:00<00:00, 2070.49it/s]


  loss: 0.3026806868293973
epoch3 : 

...

  loss: 0.0461658992053965
epoch4 : 

100%|██████████| 4/4 [00:00<00:00, 4212.21it/s]


  loss: 0.005617152034015819
epoch5 : 

100%|██████████| 4/4 [00:00<00:00, 2075.36it/s]

  loss: 0.004109187411902008
